# BR05S — Diagnosis recovery on the 1,000 Shaoxing ECGs

Use saved reports from all four generators and all five arms. No GPU or new generation is required. Run in the graph/evaluation environment with NumPy. This notebook preserves the existing frozen evaluator and its provenance checks.

**Required before scoring:** a qualified reviewer must review the frozen SNOMED-to-concept mapping against the source dictionary, including broad conduction/tachycardia categories and QT scope. The ten-case report review does not complete this separate mapping review. Do not enter an approval note merely to bypass the gate. If mapping changes are needed, stop and create a versioned target protocol; do not edit frozen targets.

With positive-only labels, evaluate recovery of documented diagnoses, not full diagnostic accuracy. Unknown diagnoses remain masked. ROUGE-L requires independent reference reports and is not computed here.


In [ ]:
from pathlib import Path
import sys, time
from collections import Counter
EXP = Path('/path/to/repository/experiment02')
sys.path.insert(0, str(EXP))
from common import load_config, parent_setup, read, sha, signature, atomic, csv_save, assert_config
CONFIG_PATH = Path('/path/to/repository/outputs/real_run_001/experiment02_v1/context_fit/v1/shaoxing/config.json')
CFG = load_config(CONFIG_PATH)
parent_setup(CFG)
assert CFG['cohort'] == 'shaoxing', 'Select the Shaoxing child config'
assert_config(CFG)
RUN = Path(CFG['run_dir'])
TARGETS_PATH = Path(CFG['shaoxing_cohort']) / 'targets.json'
TOKEN_LIMITS = {'pulse': 384, 'llama_ecg': 384, 'llama_base': 1024, 'qwen38': 1024}
FOLDERS = {m: RUN / 'generations' / 'shaoxing' / m / f'nonthinking_t{n}' for m,n in TOKEN_LIMITS.items()}
DESTINATION = RUN / 'evaluation' / 'shaoxing' / 'diagnosis_recovery_v1'
# Fill ONLY after actual mapping review; include reviewer, date, scope and decisions.
MAPPING_REVIEW_NOTE = ''
MAX_RUN_HOURS = 30.0
REPEATS = 2000
print('Results:', DESTINATION)
for m,p in FOLDERS.items(): print(m, p)


## Audit targets and generation coverage
This cell can run before mapping review. It verifies the frozen target signature and audits only the 1,000 selected queries. It does not approve the clinical mapping. A missing/unknown label is never converted to zero.


In [ ]:
from evaluation import load_results
from candidate_preparation import ECG_RULES
manifest_path = RUN / 'packets' / 'shaoxing' / 'manifest.json'
manifest = read(manifest_path)
manifest['_file_sha256'] = sha(manifest_path)
query_ids = manifest['query_ids']
assert len(query_ids) == len(set(query_ids)) == 1000, 'Expected 1,000 unique queries'
assert set(manifest['arms']) == {'sensor_only','forward_pool','bidirectional_pool','bidirectional_graph','bidirectional_shuffled_graph'}
protocol = read(Path(CFG['shaoxing_cohort']) / 'cohort_protocol.json')
raw_targets = read(TARGETS_PATH)
assert signature(raw_targets) == protocol['targets_sha256'], 'Frozen targets changed'
targets = {r['id']: r for r in raw_targets}
assert len(targets) == len(raw_targets), 'Duplicate target IDs'
assert set(query_ids) <= set(targets), 'Missing selected targets'
concepts = sorted(set(protocol['mapping'].values()))
assert set(concepts) <= set(ECG_RULES), 'Target concept absent from report parser'
label_counts = {c: Counter() for c in concepts}
positive_cases = 0
for q in query_ids:
    findings = targets[q]['findings']
    assert set(findings) <= set(concepts), (q, 'Unexpected target concepts')
    for c in concepts:
        value = findings.get(c)
        assert value is None or (type(value) is int and value in (0,1)), (q,c,value)
        label_counts[c]['unknown' if value is None else 'positive' if value == 1 else 'negative'] += 1
    positive_cases += any(v == 1 for v in findings.values())
label_rows = [dict(concept=c, **{k:label_counts[c][k] for k in ('positive','negative','unknown')}) for c in concepts]
assert not sum(r['negative'] for r in label_rows), 'This notebook assumes positive-only targets; review the scoring plan for negative labels'
assert positive_cases > 0, 'No evaluable positive targets'
coverage_rows = []
for model, folder in FOLDERS.items():
    p = read(folder / 'protocol.json')
    assert p['model_key'] == model and p['enable_thinking'] is False
    assert p['max_new_tokens'] == TOKEN_LIMITS[model]
    saved = load_results(folder, manifest)
    for arm in manifest['arms']:
        statuses = Counter(saved.get((q,arm),{}).get('status','not_attempted') for q in query_ids)
        coverage_rows.append(dict(model=model,arm=arm,expected=1000,ok=statuses['ok'],
            failed=sum(v for k,v in statuses.items() if k not in ('ok','not_attempted')),
            not_attempted=statuses['not_attempted']))
DESTINATION.mkdir(parents=True, exist_ok=True)
csv_save(DESTINATION / 'target_label_coverage.csv', label_rows)
csv_save(DESTINATION / 'generation_coverage.csv', coverage_rows)
atomic(DESTINATION / 'target_audit.json', dict(queries=1000,positive_label_queries=positive_cases,
    zero_positive_queries=1000-positive_cases,targets_sha256=sha(TARGETS_PATH),
    frozen_mapping=protocol['mapping'],negative_concepts=protocol.get('negative_concepts',[]),
    frozen_mapping_review_note=protocol.get('mapping_review_note'),clinical_mapping_approved_by_this_audit=False))
print('Queries with documented mapped diagnoses:', positive_cases)
for r in label_rows: print(r)
for r in coverage_rows: print(r)


## Evaluate within each generator
All five arms are compared on that model's complete-query subset, with pairwise-complete comparisons and completion-sensitive F1 also saved. Failures are retained in coverage, not silently discarded. Different models can have different denominators, so these results alone do not support direct model rankings.

Rerun unchanged after an interruption. The existing evaluator caches scoring and bootstrap results. Do not change reports, targets, review note or settings mid-run; use a new destination version for a revised protocol. A STOP file left by a prior run must be deliberately cleared before resuming.


In [ ]:
from evaluation import evaluate
if not MAPPING_REVIEW_NOTE.strip() or MAPPING_REVIEW_NOTE.lower().startswith('draft'):
    raise ValueError('Complete the clinical SNOMED/concept mapping review, then enter its factual documentation in MAPPING_REVIEW_NOTE. Ten-case report adjudication is a separate task.')
started = time.monotonic()
for model, folder in FOLDERS.items():
    remaining = MAX_RUN_HOURS - (time.monotonic()-started)/3600
    if remaining <= 0:
        print('Allocation budget reached; rerun this cell to resume.'); break
    result = evaluate(CFG, {model:folder}, TARGETS_PATH, DESTINATION/model,
        reference_texts={}, allow_partial=True, mapping_review_note=MAPPING_REVIEW_NOTE,
        max_run_hours=remaining, repeats=REPEATS)
    print(model, 'complete:', result['complete'], 'matched queries:', result.get('matched_queries'))
    if not result['complete']:
        print('Stopped; resume this cell with identical settings.'); break


## Export diagnosis-recovery tables
Positive-label recall = recovered documented positives / documented positives; omission = 1 − recall. With no known negatives, masked F1 = 2R/(1+R) for each evaluable report. It is not a measure of false-positive control. Unsupported additional diagnoses require clinical review; unknown-reference assertions are not automatically errors.

The original evaluator's precision, Hamming accuracy and reference-discordant positive rate remain in its raw outputs, but are not evidence of specificity or hallucination reduction in this positive-only dataset. Zero-positive queries have undefined recovery and are excluded from its denominator. The following tables keep those limitations explicit.

Graph versus bidirectional pool isolates the incremental graph context. Graph versus shuffled graph tests whether the graph structure matters. Other contrasts are supportive. Combined Holm adjustment below covers four model-specific graph-versus-pool recall tests; all analyses remain exploratory and do not undo prior validation-based selection.


In [ ]:
from evaluation import holm
summary, comparisons, completion = [], [], []
for model in FOLDERS:
    path = DESTINATION/model/'results.json'
    if not path.exists() or not read(path)['complete']:
        raise RuntimeError('Finish evaluation for all models before export: '+model)
    result = read(path)
    for r in result['summary']:
        if r.get('analysis') == 'completion_sensitive_f1':
            completion.append(r); continue
        summary.append(dict(model=model,arm=r['arm'],matched_reports=r['reports'],
            evaluable_positive_queries=r['n_recall'],mean_positive_label_recall=r['mean_recall'],
            mean_omission_rate=r['mean_omission_rate'],mean_masked_f1=r['mean_f1'],
            micro_masked_f1=r['micro_f1'],unknown_reference_positive_assertions=r['unknown_reference_positive_assertions']))
    for r in result['comparisons']:
        if r['metric'] not in ('recall','omission_rate','f1','micro_f1'): continue
        row = {k:v for k,v in r.items() if k not in ('ci95','p_holm')}
        row['metric'] = {'recall':'positive_label_recall','f1':'masked_f1','micro_f1':'micro_masked_f1'}.get(r['metric'],r['metric'])
        row['ci95_lower'],row['ci95_upper'] = r.get('ci95') or [None,None]
        comparisons.append(row)
primary = [r for r in comparisons if r['metric']=='positive_label_recall' and r['subset']=='all_arm_complete' and r['contrast']=='bidirectional_graph_minus_bidirectional_pool']
adjusted = holm({r['model_family']:r['p_value'] for r in primary})
for r in primary: r['p_holm_four_model_recall_tests'] = adjusted[r['model_family']]
csv_save(DESTINATION/'diagnosis_recovery_summary.csv',summary)
csv_save(DESTINATION/'diagnosis_recovery_paired_comparisons.csv',comparisons)
csv_save(DESTINATION/'completion_sensitive_masked_f1.csv',completion)
print('Saved analysis tables:', DESTINATION)
for r in summary: print(r)


## What to download
Download the entire `diagnosis_recovery_v1` folder, including audits, four model folders, protocols and coverage. Each model folder contains `per_report.csv`, `summary.csv`, `paired_comparisons.csv` and `results.json`.

Report denominators and generation failures alongside gains. Disclose that base Llama uses the later 1,024-token run while PULSE/ECG-Instruct use 384; preserve the unused 384-token base-Llama run. The current regex parser needs an error audit on clinician-reviewed cases. Keep the single-reviewer calibration results separate from the 1,000-case coded-diagnosis evaluation. No reference reports are supplied, so do not report ROUGE-L as zero or generate synthetic reference reports from codes.
